# 비교 차트 — 막대그래프 · 그룹 막대 · 누적 막대 · 파이 차트


**설명 전용 자료:** 개념·예제·주의점을 Markdown 셀로 제공합니다. 연습문제와 빈 풀이 셀은 같은 이름의 `_Practice.ipynb`에 있습니다.

**학습 목표:** 범주형 데이터의 크기를 비교하기 위해 수직·수평 막대그래프, 정렬 원칙, 그룹 막대(`sns.barplot`과 에러바), 빈도 막대(`sns.countplot`), 누적 막대, 파이 차트를 상황에 맞게 선택하고 작성합니다.

## 처음 배우는 분을 위한 안내

빈도는 등장 횟수, 합계는 금액을 모두 더한 값, 평균은 합계를 관측 수로 나눈 값입니다. bar는 이미 계산한 값을 막대로, countplot은 행 수를 자동 집계해 그립니다. barplot 기본 높이는 합계가 아닌 평균입니다. hue는 그룹별 색상입니다.

**학습 순서:** 문제 1~6은 필수 기초, 7~12는 선택 도전입니다. 문제 11은 9번의 all_channel을 사용합니다. 에러바는 평균 추정의 불확실성을 나타내며 개별 주문의 최소·최대가 아닙니다.

셀은 실행 단위입니다. Shift+Enter로 한 셀씩 위에서 아래로 실행합니다. 화면에 보이는 ```python``` 회색 코드 블록은 설명 문서이므로 실행하려면 새 코드 셀에 복사하세요. 준비 셀은 데이터만 만들며 문제의 답은 포함하지 않습니다. 빈 풀이 셀은 오류 없이 실행되지만 아직 결과를 계산한 것은 아닙니다. **완료할 때는 계산 결과 하나와 그 의미 한 문장을 함께 남기세요.**

그래프 창이 비어 있으면 데이터 개수와 결측 제거 결과를 먼저 확인하세요. 한글이 네모로 나오면 글꼴 설정 문제이며 계산 오류와 구분합니다.

## 실행 환경과 예제 데이터

아래 가상 데이터를 공통으로 사용합니다. 코드 블록은 읽기용이며 실행하려면 별도 코드 셀에 복사하세요.

```python
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 온라인·오프라인 주문 가상 데이터
online = pd.DataFrame({
    'order_id': range(101, 113),
    'product_category': ['Tech', 'Food', 'Tech', 'Book', 'Food', 'Tech',
                         'Food', 'Book', 'Tech', 'Food', 'Book', 'Tech'],
    'region': ['South', 'North', 'South', 'East', 'North', 'South',
               'East', 'North', 'South', 'West', 'North', 'South'],
    'payment_method': ['Card', 'Wallet', 'Card', 'Card', 'COD', 'Card',
                       'Wallet', 'Card', 'Wallet', 'COD', 'Card', 'Card'],
    'revenue': [320, 80, 450, 120, 90, 500, 110, 150, 380, 70, 130, 400]
})

offline = pd.DataFrame({
    'order_id': range(201, 209),
    'product_category': ['Tech', 'Food', 'Book', 'Tech', 'Food', 'Book', 'Tech', 'Food'],
    'region': ['South', 'South', 'North', 'East', 'North', 'East', 'West', 'South'],
    'payment_method': ['Card', 'COD', 'Card', 'Card', 'COD', 'Wallet', 'Card', 'COD'],
    'revenue': [280, 120, 160, 340, 140, 180, 300, 100]
})

# 고객 마스터 및 문의 유형 가상 데이터
master = pd.DataFrame({
    'customer_id': [f'C{i:02d}' for i in range(1, 13)],
    'membership_tier': ['Bronze', 'Silver', 'Gold', 'Bronze', 'Bronze', 'Silver',
                        'VIP', 'Bronze', 'Silver', 'Gold', 'Bronze', 'Silver']
})

inquiries = pd.DataFrame({
    'inquiry_type': [
        '상품 배송 일정 및 지연 문의',
        '회원 등급 및 포인트 적립 문의',
        '주문 취소 및 환불 절차 문의',
        '결제 오류 및 영수증 발급 문의'
    ],
    'count': [45, 20, 38, 30]
})
```

## 막대그래프(`plt.bar`) vs 히스토그램(`plt.hist`) 구분

모양은 비슷해 보이지만 두 차트는 데이터의 종류와 목적이 완전히 다릅니다.

| 구분 | 막대그래프 (`plt.bar`, `sns.barplot`) | 히스토그램 (`plt.hist`, `sns.histplot`) |
| --- | --- | --- |
| 대상 데이터 | **범주형** (카테고리, 지역, 등급 등) | **연속형 수치** (매출액, 나이, 배송 기간 등) |
| X축의 의미 | 독립된 항목 이름 (순서를 바꿔도 의미 유지) | 연속된 수치 구간 (`bins`, 순서 변경 불가) |
| 막대 사이 간격 | **있음** (서로 다른 개별 항목임을 표현) | **없음** (연속된 구간임을 표현) |
| 주요 목적 | 범주 간 값 크기(합계·평균·건수) 비교 | 데이터가 어디에 몰려 있는지 분포 형태 파악 |

## 수직/수평 막대 선택과 정렬 원칙

범주 간 비교 차트는 **값의 크기 순으로 정렬**해야 독자가 1위와 하위 항목을 즉시 파악할 수 있습니다. (단, 연도·월이나 등급 순서처럼 고유한 순서가 있는 범주는 원래 순서를 유지합니다.)

| 구분 | 수직 막대그래프 (`plt.bar`) | 수평 막대그래프 (`plt.barh`) |
| --- | --- | --- |
| 적합한 상황 | 범주 이름이 짧고 항목 수가 적을 때 | 범주 이름이 길거나 항목 수가 많을 때 |
| 권장 정렬 | 내림차순 (`ascending=False`) → 왼쪽부터 큰 값 | 오름차순 (`ascending=True`) → **위쪽**에 가장 큰 값 배치 |

```python
# 수평 막대그래프: 오름차순 정렬해야 가장 큰 막대가 맨 위에 그려집니다.
inq_sorted = inquiries.sort_values('count', ascending=True)
plt.figure(figsize=(7, 3.5))
plt.barh(inq_sorted['inquiry_type'], inq_sorted['count'], color='steelblue')
plt.xlabel('문의 건수')
plt.title('고객 문의 유형별 건수 (많은 순)')
plt.tight_layout()
plt.show()
```

## 그룹 막대그래프(`sns.barplot`)와 에러바(Error Bar), `countplot`

- **`sns.countplot(data=df, x='범주열', hue='그룹열')`**: 별도의 `groupby`나 `value_counts()` 집계 없이 범주별 행 개수(빈도)를 자동으로 세어 막대로 그려줍니다.
- **`sns.barplot(data=df, x='범주열', y='수치열', hue='그룹열')`**: 각 범주(및 `hue` 하위 그룹)의 **평균값**을 막대 높이로 그리고, 막대 끝에 **에러바(Error Bar, 기본 95% 신뢰구간)**를 표시합니다.
  - 에러바가 짧으면 표본 값들이 고르게 모여 평균 추정이 비교적 정밀하다는 뜻이고, 에러바가 길면 표본 수가 적거나 값의 편차가 크다는 뜻입니다.
  - 단순 평균 막대 높이만 깔끔하게 비교하고 싶을 때는 `errorbar=None`을 지정합니다.

```python
plt.figure(figsize=(7, 4))
sns.barplot(
    data=online,
    x='region',
    y='revenue',
    hue='payment_method',
    errorbar=None
)
plt.title('지역 및 결제수단별 평균 주문 매출')
plt.show()
```

## 누적 막대그래프(Stacked Bar)와 파이 차트(`plt.pie`)

- **누적 막대그래프(`df.plot(kind='bar', stacked=True)`)**: 그룹이 많아 나란히 놓기 복잡할 때, 하나의 막대 안에 하위 구성 요소를 쌓아 **전체 합계와 구성 비율**을 동시에 보여줍니다. 단, 맨 아래층을 제외한 위쪽 구간들은 시작 기준선이 서로 달라 미세한 크기 비교는 어렵습니다.
- **원형 차트(`plt.pie(values, labels=..., autopct='%.1f%%', explode=...)`)**: 전체(`100%`)에서 각 범주가 차지하는 **비중(점유율)**을 직관적으로 보여줍니다. 항목이 5개를 넘어가거나 값 차이가 작으면 각도 비교가 어려우므로 막대그래프가 더 정확합니다.

## 선택 도전: 건수와 합계가 다른 질문임을 확인

**필수 아님:** 위의 기본 문제를 먼저 완료하세요. 새 파일을 구하거나 준비 셀의 변수를 교체하지 않습니다. 아래 작은 입력만 별도 코드 셀에 복사해 사용합니다.

### 입력

```python
import pandas as pd
extra_orders = pd.DataFrame({'category':['A','A','B','B'],'revenue':[100,200,50,250]})
```

### 진행 순서

1. category별 행 수와 revenue 합계를 따로 구합니다.
2. 합계를 막대그래프로 그리고 X축 Category, Y축 Revenue라고 표시합니다.
3. 합계가 같더라도 개별 주문 금액은 같은지 설명합니다.

**완료 기준:** A와 B 모두 2건, 매출 합계 300씩. B의 개별 금액은 50·250으로 A의 100·200과 다름.

### 선택 도전 풀이

```python
import matplotlib.pyplot as plt
extra_summary=extra_orders.groupby('category').agg(order_count=('revenue','size'),total_revenue=('revenue','sum'))
print(extra_summary)
plt.bar(extra_summary.index,extra_summary['total_revenue'])
plt.xlabel('Category');plt.ylabel('Revenue');plt.title('Two Category Totals');plt.show()
assert extra_summary['total_revenue'].tolist()==[300,300]
```

## 복습 순서

1. 분석 대상 열이 범주형(막대그래프)인지 연속형(히스토그램)인지 먼저 구분합니다.
2. `plt.bar`는 사전 집계가 필요하고, `sns.countplot`(빈도)과 `sns.barplot`(평균)은 원본 데이터프레임을 바로 넣을 수 있다는 차이를 확인합니다.
3. 수직 막대는 내림차순(`ascending=False`), 수평 막대는 오름차순(`ascending=True`)으로 정렬해 1위 항목이 가장 눈에 띄게 배치합니다.
4. `hue`를 사용한 그룹 막대와 누적 막대, 파이 차트의 장단점을 비교합니다.